# 🌊 Sonaris — Multi-Class Lite Model Training (YOLO11)
### Train on the 4-Class Seabed Sonar Dataset

This notebook trains the **Sonaris Lite Model (YOLO11)** on the **SeabedObjects-KLSG** dataset across **4 classes**:
- `0: aircraft` (Downed plane wreckage on seabed)
- `1: fish` (Acoustic fish swarms)
- `2: other` (Seabed debris, man-made structures, pipelines, containers)
- `3: shipwreck` (Sunken ships, hulls, and maritime ruins)

⏱️ **Estimated Training Time**: ~15-20 minutes on Google Colab (Free T4 GPU).

## Step 1: Check GPU Acceleration
Make sure **GPU** is active (**Runtime → Change runtime type → T4 GPU**).

In [ ]:
!pip install ultralytics roboflow -q

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: Running on CPU. Please switch to GPU: Runtime -> Change runtime type -> T4 GPU")

## Step 2: Download the 4-Class Sonar Dataset

### 👉 Enter your free Roboflow API Key below:
1. Go to [app.roboflow.com/settings/api](https://app.roboflow.com/settings/api) (Sign in free with Google).
2. Copy your **Private API Key** and paste it into `ROBOFLOW_API_KEY` below.

*(Alternatively, if you already downloaded the ZIP from [Roboflow Universe](https://universe.roboflow.com/object-detect-ury2h/sonar_detect/dataset/1), upload it to Colab's left sidebar file panel and run `!unzip -q *.zip -d dataset`)*

In [ ]:
import os
import glob

# Paste your Roboflow API key here (from app.roboflow.com/settings/api)
ROBOFLOW_API_KEY = ""  # <-- PASTE YOUR KEY HERE

data_yaml_path = None

if ROBOFLOW_API_KEY.strip():
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    print("Downloading SeabedObjects sonar dataset...")
    project = rf.workspace("object-detect-ury2h").project("sonar_detect")
    dataset = project.version(1).download("yolov8")
    data_yaml_path = os.path.join(dataset.location, "data.yaml")
else:
    # Check if a zip file was uploaded manually
    zip_files = glob.glob("*.zip")
    if zip_files:
        print(f"Found uploaded zip: {zip_files[0]}. Extracting...")
        !unzip -q "{zip_files[0]}" -d dataset
        data_yaml_path = "dataset/data.yaml"
    else:
        print("⚠️ Please provide your Roboflow API key above, OR upload your dataset zip file to Colab's file panel.")

if data_yaml_path and os.path.exists(data_yaml_path):
    print(f"✅ Dataset verified! Path: {data_yaml_path}")
    with open(data_yaml_path) as f:
        print(f.read())
else:
    print("❌ Dataset not ready yet. Please paste your API key above and run this cell again.")

## Step 3: Train YOLO11 Nano on the 4 Classes

In [ ]:
import torch
from ultralytics import YOLO
import os

# Verify data.yaml is found
yaml_candidates = glob.glob("**/data.yaml", recursive=True)
if not yaml_candidates:
    raise FileNotFoundError("data.yaml not found! Please run Step 2 first to download the dataset.")

yaml_file = os.path.abspath(yaml_candidates[0])
print(f"Using dataset configuration: {yaml_file}")

# Load lightweight base YOLO11 model
model = YOLO('yolo11n.pt')

# Train on the 4-class sonar dataset
results = model.train(
    data=yaml_file,
    epochs=100,            # 100 epochs for optimal convergence
    imgsz=640,
    batch=16,
    patience=25,           # Early stopping
    device=(0 if torch.cuda.is_available() else 'cpu'),              # GPU acceleration
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.001,
    cos_lr=True,
    augment=True,
    mosaic=0.5,
    project='sonaris_runs',
    name='sonaris_multiclass_lite',
    exist_ok=True
)

print("\n🎉 Training complete!")


## Step 4: Evaluate the 4 Classes

In [ ]:
# Validate on the test split
metrics = model.val(data=yaml_file, split='test')

print("\n--- 📊 Final Validation Metrics ---")
print(f"Overall mAP@50:    {metrics.box.map50:.3f}")
print(f"Overall mAP@50-95: {metrics.box.map:.3f}")

# Print per-class metrics
for i, cls_name in enumerate(model.names.values()):
    if i < len(metrics.box.maps):
        print(f"  • Class '{cls_name}': AP@50 = {metrics.box.maps[i]:.3f}")

## Step 5: Download the Trained Model Weights
Download `best.pt` to your computer and replace `Sonaris/models/yolo11n_seg_best.pt`!

In [ ]:
from google.colab import files
import os

weight_path = 'sonaris_runs/sonaris_multiclass_lite/weights/best.pt'
if os.path.exists(weight_path):
    print(f"Downloading {weight_path}...")
    files.download(weight_path)
else:
    print("Weights file not found yet. Make sure Step 3 finished successfully.")